# T3P Packet Inspector

In [ ]:
import struct
import re
import os
import mmap
import numpy as np

def inspect_t3p_range(filepath, start_packet, end_packet):
    print(f"--- T3P EXACT PACKET INSPECTOR ---")
    print(f"File: {os.path.basename(filepath)}")
    print(f"Printing records #{start_packet} through #{end_packet}...")
    print(f"Scanning the entire file for global totals (this may take a moment for massive files)...\n")
    
    # This regex matches the tab-separated ASCII lines (The HW Triggers)
    hw_trigger_pattern = re.compile(rb'(?:\d+\t)+\d+\r?\n')
    
    with open(filepath, 'rb') as f:
        # Use memory-mapping to scan massive files instantly without overloading RAM
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        
        # Pre-scan for all ASCII HW Trigger injections
        hw_trigger_traps = list(hw_trigger_pattern.finditer(mm))
        
        offset = 0
        record_count = 0
        header_printed = False
        
        # Global Counters for the entire file
        total_hw_triggers = 0
        total_photon_hits = 0
        chip_0_hits = 0  # overflow == 0
        chip_1_hits = 0  # overflow != 0
        
        # Loop over the ENTIRE file to get true global totals
        while offset < len(mm):
            
            # 1. CHECK FOR ASCII HW TRIGGER INJECTIONS
            if hw_trigger_traps and offset <= hw_trigger_traps[0].start() < offset + 16:
                trap = hw_trigger_traps.pop(0)
                
                # Jump over any corrupted/cut-off bytes directly to the text
                offset = trap.start()
                
                # The text string itself counts as one record in the sequence
                record_count += 1
                total_hw_triggers += 1
                
                text_bytes = mm[trap.start() : trap.end()]
                
                if start_packet <= record_count <= end_packet:
                    text_str = text_bytes.decode('ascii', errors='ignore').strip().replace('\t', ' \\t ')
                    print(f"\n[{record_count:08d}] [OFFSET: {trap.start():08X}] ⚡ HW TRIGGER (TEXT) | Length: {len(text_bytes):02d} bytes | Text: '{text_str}'\n")
                    header_printed = False # Reset header so it reprints after the text interruption
                
                # Resume standard 16-byte reading immediately after the text ends
                offset = trap.end()
                continue
                
            # 2. READ THE STANDARD 16-BYTE BINARY PACKET (PHOTON HITS)
            if offset + 16 > len(mm):
                break 
                
            packet = mm[offset : offset + 16]
            
            # Unpack all 5 variables from the C-Struct
            matrixIdx, toa, overflow, ftoa, tot = struct.unpack('<IQBBH', packet)
            record_count += 1
            
            # Global Tally logic
            total_photon_hits += 1
            if overflow == 0:
                chip_0_hits += 1
            else:
                chip_1_hits += 1
            
            # 3. PRINT THE PACKET ONLY IF IT FALLS IN THE RANGE
            if start_packet <= record_count <= end_packet:
                
                if not header_printed:
                    print(f"{'RECORD #':<10} | {'BYTE OFFSET':<11} | {'PACKET TYPE':<14} | {'matrixIdx':<10} | {'ToA':<12} | {'ToT':<5} | {'fToA':<4} | {'Overflow (Chip)'}")
                    print("-" * 95)
                    header_printed = True
                    
                print(f"[{record_count:08d}] | {offset:08X}    | 🔵 PHOTON HIT   | {matrixIdx:<10} | {toa:<12} | {tot:<5} | {ftoa:<4} | {overflow}")
                
            offset += 16
            
        mm.close()
        
    print("\n--- INSPECTION COMPLETE ---")
    print(f"► Total HW triggers (Text):           {total_hw_triggers}")
    print(f"► Total photon hits (Binary):         {total_photon_hits}")
    print(f"  ├─ Hits on Chip 0 (overflow == 0):  {chip_0_hits}")
    print(f"  └─ Hits on Chip 1+ (overflow != 0): {chip_1_hits}")
    
# ==========================================
# EXECUTION
# ==========================================
t3p_file = r"G:\האחסון שלי\X-Ray-IFM\Test Files\Sync_test\sync_test_25.5_r0.t3p"

# Change these two numbers to whatever range you want to print!
start = 0
end = 5

inspect_t3p_range(t3p_file, start, end)

--- T3P EXACT PACKET INSPECTOR ---
File: sync_test_25.5_r0.t3p
Printing records #0 through #5...
Scanning the entire file for global totals (this may take a moment for massive files)...


[00000001] [OFFSET: 00000000] ⚡ HW TRIGGER (TEXT) | Length: 14 bytes | Text: '0 \t 0 \t 62 \t 0 \t 0 \t 10'


[00000002] [OFFSET: 0000000E] ⚡ HW TRIGGER (TEXT) | Length: 18 bytes | Text: '1 \t 0 \t 62 \t 0 \t 49152 \t 10'

RECORD #   | BYTE OFFSET | PACKET TYPE    | matrixIdx  | ToA          | ToT   | fToA | Overflow (Chip)
-----------------------------------------------------------------------------------------------
[00000003] | 00000020    | 🔵 PHOTON HIT   | 37042      | 6536         | 14    | 7    | 0
[00000004] | 00000030    | 🔵 PHOTON HIT   | 37041      | 6536         | 28    | 9    | 0
[00000005] | 00000040    | 🔵 PHOTON HIT   | 36786      | 6536         | 23    | 8    | 0

--- INSPECTION COMPLETE ---
► Total HW triggers (Text):           1203
► Total photon hits (Binary):         19153665
  ├─

TypeError: 'int' object is not iterable

In [26]:
import struct
import re
import os
import mmap
import numpy as np

def inspect_and_parse_t3p(filepath, start_packet, end_packet):
    print(f"--- T3P EXACT PACKET INSPECTOR & PARSER ---")
    print(f"File: {os.path.basename(filepath)}")
    print(f"Printing records #{start_packet} through #{end_packet}...")
    print(f"Scanning the entire file to build NumPy arrays...\n")
    
    # This regex matches the tab-separated ASCII lines (The HW Triggers)
    hw_trigger_pattern = re.compile(rb'(?:\d+\t)+\d+\r?\n')
    
    # Lists to accumulate data before NumPy conversion (much faster than appending to NumPy arrays in a loop)
    photon_list = []
    trigger_list = []
    
    with open(filepath, 'rb') as f:
        # Use memory-mapping to scan massive files instantly without overloading RAM
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        
        # Pre-scan for all ASCII HW Trigger injections
        hw_trigger_traps = list(hw_trigger_pattern.finditer(mm))
        
        offset = 0
        record_count = 0
        header_printed = False
        
        # Global Counters for the entire file
        chip_0_hits = 0  # overflow == 0
        chip_1_hits = 0  # overflow != 0
        
        # Loop over the ENTIRE file
        while offset < len(mm):
            
            # 1. CHECK FOR ASCII HW TRIGGER INJECTIONS
            if hw_trigger_traps and offset <= hw_trigger_traps[0].start() < offset + 16:
                trap = hw_trigger_traps.pop(0)
                
                # Jump over any corrupted/cut-off bytes directly to the text
                offset = trap.start()
                
                # The text string itself counts as one record in the sequence
                record_count += 1
                
                text_bytes = mm[trap.start() : trap.end()]
                text_str = text_bytes.decode('ascii', errors='ignore').strip()
                
                # Store the DAQ pulse packet
                trigger_list.append(text_str)
                
                if start_packet <= record_count <= end_packet:
                    display_str = text_str.replace('\t', ' \\t ')
                    print(f"\n[{record_count:08d}] [OFFSET: {trap.start():08X}] ⚡ HW TRIGGER (TEXT) | Length: {len(text_bytes):02d} bytes | Text: '{display_str}'\n")
                    header_printed = False # Reset header so it reprints after the text interruption
                
                # Resume standard 16-byte reading immediately after the text ends
                offset = trap.end()
                continue
                
            # 2. READ THE STANDARD 16-BYTE BINARY PACKET (PHOTON HITS)
            if offset + 16 > len(mm):
                break 
                
            packet = mm[offset : offset + 16]
            
            # Unpack all 5 variables from the C-Struct
            matrixIdx, toa, overflow, ftoa, tot = struct.unpack('<IQBBH', packet)
            record_count += 1
            
            # Store the photon hit as a tuple
            photon_list.append((matrixIdx, toa, overflow, ftoa, tot))
            
            # Global Tally logic
            if overflow == 0:
                chip_0_hits += 1
            else:
                chip_1_hits += 1
            
            # 3. PRINT THE PACKET ONLY IF IT FALLS IN THE RANGE
            if start_packet <= record_count <= end_packet:
                
                if not header_printed:
                    print(f"{'RECORD #':<10} | {'BYTE OFFSET':<11} | {'PACKET TYPE':<14} | {'matrixIdx':<10} | {'ToA':<12} | {'ToT':<5} | {'fToA':<4} | {'Overflow (Chip)'}")
                    print("-" * 95)
                    header_printed = True
                    
                print(f"[{record_count:08d}] | {offset:08X}    | 🔵 PHOTON HIT   | {matrixIdx:<10} | {toa:<12} | {tot:<5} | {ftoa:<4} | {overflow}")
                
            offset += 16
            
        mm.close()

    # ==========================================
    # NUMPY ARRAY CONVERSION
    # ==========================================
    # Define a structured dtype that perfectly matches your 16-byte binary structure
    photon_dtype = np.dtype([
        ('matrixIdx', np.uint32),
        ('toa', np.uint64),
        ('overflow', np.uint8),
        ('ftoa', np.uint8),
        ('tot', np.uint16)
    ])
    
    photon_array = np.array(photon_list, dtype=photon_dtype)
    trigger_array = np.array(trigger_list)
        
    print("\n--- INSPECTION COMPLETE ---")
    print(f"► Total HW triggers (Text):           {len(trigger_array)}")
    print(f"► Total photon hits (Binary):         {len(photon_array)}")
    print(f"  ├─ Hits on Chip 0 (overflow == 0):  {chip_0_hits}")
    print(f"  └─ Hits on Chip 1+ (overflow != 0): {chip_1_hits}")
    
    print("\n--- MATRIX INDEX ANALYSIS ---")
    if len(photon_array) > 0:
        max_idx = np.max(photon_array['matrixIdx'])
        print(f"► Maximum matrixIdx found: {max_idx}")
        
        # 256 * 256 = 65,536 pixels per chip. Indices run 0 to 65,535.
        if max_idx > 65535:
            print("  └─ Conclusion: Index exceeds 65,535. The camera uses a GLOBAL indexing scheme across both chips (e.g., 256x512).")
        else:
            print("  └─ Conclusion: Index is <= 65,535. The camera uses SEPARATE indexing per chip (resets to 0 for the second chip).")
    else:
        print("► No photon hits found to analyze.")

    print("\n--- OVERFLOW ANALYSIS ---")  
    if len(photon_array) > 0:
        max_overflow = np.max(photon_array['overflow'])
        print(f"► Maximum overflow found: {max_overflow}")

    return photon_array, trigger_array

# ==========================================
# EXECUTION
# ==========================================
t3p_file = r"G:\האחסון שלי\X-Ray-IFM\Test Files\Sync_test\sync_test_25.5_r1.t3p"

start = 0
end = 50

# Capture the returned arrays
photons, triggers = inspect_and_parse_t3p(t3p_file, start, end)

# You can now manipulate 'photons' and 'triggers' directly as NumPy arrays!
# Example: 
# print(photons['toa'])       # prints all Time of Arrival values
# print(photons['matrixIdx']) # prints all matrix indices

--- T3P EXACT PACKET INSPECTOR & PARSER ---
File: sync_test_25.5_r1.t3p
Printing records #0 through #50...
Scanning the entire file to build NumPy arrays...


[00000001] [OFFSET: 00000000] ⚡ HW TRIGGER (TEXT) | Length: 18 bytes | Text: '0 \t 0 \t 62 \t 0 \t 49152 \t 10'


[00000002] [OFFSET: 00000012] ⚡ HW TRIGGER (TEXT) | Length: 14 bytes | Text: '1 \t 0 \t 62 \t 0 \t 0 \t 10'

RECORD #   | BYTE OFFSET | PACKET TYPE    | matrixIdx  | ToA          | ToT   | fToA | Overflow (Chip)
-----------------------------------------------------------------------------------------------
[00000003] | 00000020    | 🔵 PHOTON HIT   | 37129      | 14803        | 9     | 15   | 0
[00000004] | 00000030    | 🔵 PHOTON HIT   | 37386      | 14803        | 10    | 16   | 0
[00000005] | 00000040    | 🔵 PHOTON HIT   | 37385      | 14803        | 109   | 17   | 0
[00000006] | 00000050    | 🔵 PHOTON HIT   | 115983     | 18606        | 12    | 20   | 1
[00000007] | 00000060    | 🔵 PHOTON HIT   | 115982     | 18606 

In [24]:
np.argmax(photons['overflow'])
photons['overflow'][18853480:18853490]

array([150, 150, 150, 150, 168, 168, 168, 168,   0, 255], dtype=uint8)

# .h5 Bins Inspector

In [13]:
import h5py
import numpy as np
import os

def inspect_px5_h5(filepath, start_bin, end_bin):
    print(f"--- HDF5 PX5 DATA INSPECTOR ---")
    print(f"File: {os.path.basename(filepath)}")

    try:
        with h5py.File(filepath, 'r') as f:
            # 1. Read the Metadata attributes written by the DAQ script
            bin_s = f.attrs['bin_s']
            
            # 2. Access the main dataset
            if 'px5CountsPerBin' not in f:
                print("Error: Dataset 'px5CountsPerBin' not found in file.")
                return
                
            dset = f['px5CountsPerBin']
            total_bins = dset.shape[0]
            
            print(f"Total Bins   : {total_bins:,}")
            print(f"Bin Resolution: {bin_s * 1e6:.1f} µs ({bin_s} s)")
            print(f"File Duration: {total_bins * bin_s:.2f} seconds")
            
            # We can sum the entire array instantly to find the total hits
            print(f"Total Photons: {np.sum(dset):,}") 
            print("-" * 55)
            
            # 3. Validate user input range
            if start_bin < 0: start_bin = 0
            if end_bin >= total_bins: end_bin = total_bins - 1
            if start_bin > end_bin:
                print("Invalid range selected.")
                return
            
            print(f"Scanning bins #{start_bin} to #{end_bin}...\n")
            print(f"{'BIN INDEX':<12} | {'TIME (Seconds)':<15} | {'PHOTON COUNT'}")
            print("-" * 45)
            
            # 4. Extract only the specific slice of memory requested
            counts_in_range = dset[start_bin : end_bin + 1]
            
            for i, count in enumerate(counts_in_range):
                actual_bin = start_bin + i
                time_s = actual_bin * bin_s
                
                # Add a visual flag if a photon was actually detected in this bin
                if count > 0:
                    count_str = f"{count}  <-- 🟢 HIT"
                else:
                    count_str = str(count)
                    
                print(f"[{actual_bin:08d}]   | {time_s:<15.6f} | {count_str}")
                
    except Exception as e:
        print(f"Failed to read HDF5 file: {e}")

# ==========================================
# EXECUTION
# ==========================================
# Point this to your generated DAQ file
h5_file = r"G:\האחסון שלי\ESRF IFM\esrf_px5_data_000.h5" 

# Select the range of bins you want to look at
start = 0
end   = 5

inspect_px5_h5(h5_file, start, end)

--- HDF5 PX5 DATA INSPECTOR ---
File: esrf_px5_data_000.h5
Total Bins   : 60,000,000
Bin Resolution: 1.0 µs (1e-06 s)
File Duration: 60.00 seconds
Total Photons: 64
-------------------------------------------------------
Scanning bins #0 to #5...

BIN INDEX    | TIME (Seconds)  | PHOTON COUNT
---------------------------------------------
[00000000]   | 0.000000        | 0
[00000001]   | 0.000001        | 0
[00000002]   | 0.000002        | 0
[00000003]   | 0.000003        | 0
[00000004]   | 0.000004        | 0
[00000005]   | 0.000005        | 0
